In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

### Imports

import os
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob


#---------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


#---------------------------------------
import tensorflow as tf
import keras
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers import SGD
#---------------------------------------


import warnings
warnings.filterwarnings("ignore")


# Display
from IPython.display import Image, display
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    raise SystemError('GPU device not founde')
print('Found GPU at: {}'.format(device_name))

In [ ]:
def load_df(path):
    classes, class_paths = zip(*[(label, os.path.join(path, label, image))
                                 for label in os.listdir(path) if os.path.isdir(os.path.join(path, label))
                                 for image in os.listdir(os.path.join(path, label))])

    df = pd.DataFrame({'Class Path': class_paths, 'Class': classes})
    return df

In [ ]:
df = load_df('/kaggle/input/brain-stroke-ct-dataset/Brain_Stroke_CT_Dataset')

In [ ]:
df### Function for counting images in each class
def count_images(df):
  plt.figure(figsize=(15,7))
  ax = sns.countplot(data=df, y=df['Class'], palette='rocket')

  plt.xlabel('')
  plt.ylabel('')
  plt.title('Count of images in each class')
  ax.bar_label(ax.containers[0])
  plt.show()

In [ ]:
# Train: 81.7%, Validation: 9.1%, Test: 9.2%

train_df, test_val_df = train_test_split(df, train_size=5842 / 7153, random_state=42, stratify=df['Class'])

valid_df, test_df = train_test_split(test_val_df, train_size=655 / len(test_val_df), random_state=20, stratify=test_val_df['Class'])

print(f"Train size: {len(train_df)} rows")
print(f"Validation size: {len(valid_df)} rows")
print(f"Test size: {len(test_df)} rows")

In [ ]:
count_images(test_df)

In [ ]:
count_images(valid_df)

In [ ]:
batch_size = 32
img_size = (224, 224)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    brightness_range=(0.8, 1.2)
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_dataframe(
    train_df, 
    x_col='Class Path',
    y_col='Class', 
    batch_size=batch_size,
    target_size=img_size
)

valid_gen = valid_datagen.flow_from_dataframe(
    valid_df,
    x_col='Class Path',
    y_col='Class', 
    batch_size=batch_size,
    target_size=img_size
)

test_gen = test_datagen.flow_from_dataframe(
    test_df, 
    x_col='Class Path',
    y_col='Class', 
    batch_size=16,
    target_size=img_size,
    shuffle=False
)

In [ ]:
base_model = tf.keras.applications.Xception(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    pooling='max'
)

# froze all layers
for layer in base_model.layers:
    layer.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dropout(rate=0.3),
    Dense(128, activation='relu'),
    Dropout(rate=0.25),
    Dense(3, activation='softmax')
])

model.compile(optimizer=Adamax(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy', Precision(), Recall()])



In [ ]:
### Training Loop on GPU

with tf.device(device_name):
    # Training of top layer
    hist = model.fit(train_gen, validation_data=valid_gen, epochs=5)

In [ ]:
import time
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.metrics import Precision, Recall

# "Defroze" some convolutional blocks
# Defroze last block of VGG16
for layer in base_model.layers[-4:]:
    layer.trainable = True
    
# Compile the model with a lower learning rate for fine-tuning
model.compile(optimizer=Adamax(learning_rate=1e-4),  # lower LR for Fine-Tuning
              loss='categorical_crossentropy',
              metrics=['accuracy', Precision(), Recall()])

# Start the timer to record the total training time
start_time = time.time()

# Train the model within the specified device (if using specific device)
with tf.device(device_name):
    hist2 = model.fit(train_gen, validation_data=valid_gen, epochs=5)

# Calculate total training time
end_time = time.time()
training_time = end_time - start_time

# Print the total training time in seconds and formatted to minutes
print(f"Total Training Time: {training_time:.2f} seconds")
print(f"Total Training Time: {training_time / 60:.2f} minutes")


In [ ]:
hist = hist2

In [ ]:
train_gen_copy = train_gen
valid_gen_copy = valid_gen
test_gen_copy = test_gen

In [ ]:
#add
from sklearn.metrics import classification_report
import numpy as np

# Evaluate model
train_score = model.evaluate(train_gen_copy, verbose=1)
valid_score = model.evaluate(valid_gen_copy, verbose=1)
test_score = model.evaluate(test_gen_copy, verbose=1)

# Predict on the datasets to calculate F1 score
train_pred = model.predict(train_gen_copy, verbose=1)
valid_pred = model.predict(valid_gen_copy, verbose=1)
test_pred = model.predict(test_gen_copy, verbose=1)

# Convert predictions to binary (for binary classification) or classes for multi-class
train_pred_classes = np.argmax(train_pred, axis=1)
valid_pred_classes = np.argmax(valid_pred, axis=1)
test_pred_classes = np.argmax(test_pred, axis=1)

# Assuming your dataset has labels in a format like one-hot encoded or integer encoded
train_labels = train_gen_copy.classes
valid_labels = valid_gen_copy.classes
test_labels = test_gen_copy.classes

# F1 score calculation using classification report
train_f1 = classification_report(train_labels, train_pred_classes, output_dict=True)['weighted avg']['f1-score']
valid_f1 = classification_report(valid_labels, valid_pred_classes, output_dict=True)['weighted avg']['f1-score']
test_f1 = classification_report(test_labels, test_pred_classes, output_dict=True)['weighted avg']['f1-score']

# Print scores
print(f"Train Loss: {train_score[0]:.4f}")
print(f"Train Accuracy: {train_score[1]*100:.2f}%")
print(f"Train F1 Score: {train_f1:.4f}")
print('-' * 20)
print(f"Validation Loss: {valid_score[0]:.4f}")
print(f"Validation Accuracy: {valid_score[1]*100:.2f}%")
print(f"Validation F1 Score: {valid_f1:.4f}")
print('-' * 20)
print(f"Test Loss: {test_score[0]:.4f}")
print(f"Test Accuracy: {test_score[1]*100:.2f}%")
print(f"Test F1 Score: {test_f1:.4f}")
#Adamax